# Camada SILVER — Limpeza e Enriquecimento
Lê da Bronze, corrige tipos, remove nulos e junta as tabelas.

In [5]:
import sys
sys.path.insert(0, '/opt/spark/work-dir')
from tools.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

spark = get_spark('Silver - Transformação')
spark.sparkContext.setLogLevel('WARN')

BRONZE = '/opt/spark/work-dir/warehouse/bronze'
SILVER = '/opt/spark/work-dir/warehouse/silver'

26/06/21 02:43:03 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [6]:
# Fornecedores — limpa CNPJ
fornecedores = (
    spark.read.format('delta').load(f'{BRONZE}/fornecedores')
    .withColumn('cnpj', F.regexp_replace(F.col('cnpj'), r'[^0-9]', ''))
    .filter(F.col('fornecedor_id').isNotNull())
)
fornecedores.show(5)
print('Fornecedores:', fornecedores.count())

+-------------+--------------------+--------------+------------+--------------+----------------+-----------+
|fornecedor_id|        razao_social|          cnpj|      regiao|        cidade|         contato|   segmento|
+-------------+--------------------+--------------+------------+--------------+----------------+-----------+
|            1|      Alves e Filhos|43815092000170|         Sul|      Curitiba| (011) 0013-3890|     Livros|
|            2|Silva Rezende e F...|86374209000130|     Sudeste|Rio de Janeiro| (011) 5423-5116|     Têxtil|
|            3|        Fogaça Ltda.|94053871000117|         Sul|     Joinville|+55 21 8495 9310|  Alimentos|
|            4| Vargas da Rocha S/A|31629785000190|Centro-Oeste|        Cuiabá|    11 3419 2832|Eletrônicos|
|            5|              Garcia|48352019000123|         Sul|      Curitiba| (051) 6413 9537| Cosméticos|
+-------------+--------------------+--------------+------------+--------------+----------------+-----------+
only showing top 5 

In [7]:
# Cupons — tipa datas e adiciona coluna ativo
cupons = (
    spark.read.format('delta').load(f'{BRONZE}/cupons')
    .withColumn('validade_inicio', F.col('validade_inicio').cast(DateType()))
    .withColumn('validade_fim', F.col('validade_fim').cast(DateType()))
    .withColumn('valor', F.col('valor').cast('double'))
    .withColumn('uso_minimo', F.col('uso_minimo').cast('double'))
    .withColumn('ativo',
        (F.current_date() >= F.col('validade_inicio'))
        & (F.current_date() <= F.col('validade_fim'))
    )
    .filter(F.col('cupom_id').isNotNull())
)
cupons.show(5)
print('Cupons:', cupons.count())

+--------+--------+----------+-----+---------------+------------+---------+----------+-----+
|cupom_id|  codigo|      tipo|valor|validade_inicio|validade_fim|categoria|uso_minimo|ativo|
+--------+--------+----------+-----+---------------+------------+---------+----------+-----+
|       1|NEMO5744|percentual| 12.0|     2023-07-09|  2023-09-06|    TODAS|       0.0|false|
|       2|QUAS9416|      fixo| 24.1|     2022-04-30|  2022-10-22|   Livros|       0.0|false|
|       3|QUIS2205|percentual| 16.0|     2023-08-29|  2024-01-09|   Beleza|    217.51|false|
|       4|VOLU2550|percentual|  7.0|     2023-09-25|  2023-12-04| Esportes|       0.0|false|
|       5| CUM2569|percentual| 17.0|     2022-04-21|  2022-08-03|     Casa|       0.0|false|
+--------+--------+----------+-----+---------------+------------+---------+----------+-----+
only showing top 5 rows
Cupons: 200


In [8]:
# Fretes — filtra cancelados e tipa campos
fretes = (
    spark.read.format('delta').load(f'{BRONZE}/fretes')
    .withColumn('valor_frete', F.col('valor_frete').cast('double'))
    .withColumn('prazo_dias', F.col('prazo_dias').cast('integer'))
    .filter(F.col('status_entrega') != 'CANCELADO')
    .filter(F.col('frete_id').isNotNull())
)
fretes.show(5)
print('Fretes:', fretes.count())

+--------+--------+-----------+----------+--------------+--------------+
|frete_id|venda_id|valor_frete|prazo_dias|transportadora|status_entrega|
+--------+--------+-----------+----------+--------------+--------------+
|       1|       1|      14.28|         8|      Correios|    PREPARANDO|
|       4|       4|      27.38|         8|        Jadlog|   EM_TRANSITO|
|       5|       5|       14.5|         6|        Jadlog|      ENTREGUE|
|       6|       6|      24.53|         1|        Jadlog|      ENTREGUE|
|       7|       7|      20.16|         9| Total Express|   EM_TRANSITO|
+--------+--------+-----------+----------+--------------+--------------+
only showing top 5 rows
Fretes: 799772


In [9]:
# Pagamentos — filtra PAGO e adiciona eh_parcelado
pagamentos = (
    spark.read.format('delta').load(f'{BRONZE}/pagamentos')
    .withColumn('valor_pago', F.col('valor_pago').cast('double'))
    .withColumn('parcelas', F.col('parcelas').cast('integer'))
    .withColumn('eh_parcelado', F.col('parcelas') > 1)
    .filter(F.col('status_pagamento') == 'PAGO')
    .filter(F.col('pagamento_id').isNotNull())
)
pagamentos.show(5)
print('Pagamentos:', pagamentos.count())

+------------+--------+--------------+--------+----------+----------------+------------+
|pagamento_id|venda_id|        metodo|parcelas|valor_pago|status_pagamento|eh_parcelado|
+------------+--------+--------------+--------+----------+----------------+------------+
|           5|       5|           pix|       1|   5524.26|            PAGO|       false|
|           6|       6|        boleto|       1|   3164.03|            PAGO|       false|
|           8|       8|cartao_credito|       1|   2573.64|            PAGO|       false|
|           9|       9|cartao_credito|       9|     426.4|            PAGO|        true|
|          10|      10|           pix|       1|    293.98|            PAGO|       false|
+------------+--------+--------------+--------+----------+----------------+------------+
only showing top 5 rows
Pagamentos: 600196


In [10]:
# Clientes — limpa, tipa e enriquece com métricas de compra
vendas_agg = (
    spark.read.format('delta').load(f'{BRONZE}/vendas')
    .filter(F.col('status') == 'CONCLUIDO')
    .groupBy('cliente_id')
    .agg(
        F.count('venda_id').alias('qtd_compras'),
        F.sum('quantidade').alias('qtd_itens'),
    )
)

clientes = (
    spark.read.format('delta').load(f'{BRONZE}/clientes')
    .withColumn('documento', F.regexp_replace(F.col('documento'), r'[^0-9]', ''))
    .withColumn('data_cadastro', F.col('data_cadastro').cast(DateType()))
    .withColumn('ativo', F.col('ativo').cast('boolean'))
    .withColumn('faixa_score',
        F.when(F.col('score_credito') >= 700, 'bom')
         .when(F.col('score_credito') >= 500, 'regular')
         .otherwise('ruim')
    )
    .filter(F.col('cliente_id').isNotNull())
    .join(vendas_agg, 'cliente_id', 'left')
)
clientes.show(5)
print('Clientes:', clientes.count())

+----------+------------------+------------+--------------+--------+-------------+-------------+-----+-------------+-----------+-----------+---------+
|cliente_id|              nome|tipo_cliente|     documento|  regiao|       cidade|data_cadastro|ativo|score_credito|faixa_score|qtd_compras|qtd_itens|
+----------+------------------+------------+--------------+--------+-------------+-------------+-----+-------------+-----------+-----------+---------+
|         1|João Pedro Sampaio|          PF|   27586039186|     Sul| Porto Alegre|   2023-04-19| true|        788.9|        bom|          6|       11|
|         2|     Allana Moraes|          PF|   65931047875|     Sul|Caxias do Sul|   2023-09-16| true|        602.7|    regular|          4|        6|
|         3|     Calebe Castro|          PF|   87263495074|   Norte|    Boa Vista|   2024-02-13|false|        575.8|    regular|          8|       25|
|         4|           Azevedo|          PJ|09281746000195|     Sul|     Curitiba|   2023-01-2

In [11]:
# Produtos — tipa e adiciona fornecedor
produtos = (
    spark.read.format('delta').load(f'{BRONZE}/produtos')
    .withColumn('preco',     F.col('preco').cast('double'))
    .withColumn('estoque',   F.col('estoque').cast('integer'))
    .withColumn('disponivel', F.col('estoque') > 0)
    .join(
        fornecedores.select('fornecedor_id', 'razao_social', 'segmento'),
        'fornecedor_id', 'left'
    )
)
produtos.show(5)
print('Produtos:', produtos.count())

+-------------+----------+------------------+-----------+------+-------+-------+----------+-------------+-----------------+
|fornecedor_id|produto_id|              nome|  categoria| preco|estoque|peso_kg|disponivel| razao_social|         segmento|
+-------------+----------+------------------+-----------+------+-------+-------+----------+-------------+-----------------+
|            5|         1|     Joelheira 599|   Esportes| 228.4|    155|  10.23|      true|       Garcia|       Cosméticos|
|           48|         2|          Pneu 217| Automotivo|603.51|     60|  28.69|      true|        Porto|        Alimentos|
|           10|         3|    Carregador 168|Eletrônicos|761.55|    407|  26.89|      true|Teixeira - EI|Casa e Construção|
|            3|         4|Fundamentos de 176|     Livros|130.33|     73|  21.04|      true| Fogaça Ltda.|        Alimentos|
|            7|         5|      Macarrão 357|  Alimentos|  56.7|    371|  28.88|      true|       Borges|       Cosméticos|
+-------

In [12]:
# Vendas — filtra CONCLUIDO, junta dimensões, fretes, pagamentos e adiciona turno
vendas = (
    spark.read.format('delta').load(f'{BRONZE}/vendas')
    .withColumn('data_venda',   F.col('data_venda').cast(DateType()))
    .withColumn('quantidade',   F.col('quantidade').cast('integer'))
    .withColumn('desconto_pct', F.col('desconto_pct').cast('double'))
    .withColumn('turno',
        F.when(F.hour(F.col('hora_venda')) < 12, 'manhã')
         .when(F.hour(F.col('hora_venda')) < 18, 'tarde')
         .otherwise('noite')
    )
    .filter(F.col('status') == 'CONCLUIDO')
    .join(produtos.select('produto_id', 'preco', 'nome', 'categoria'), 'produto_id')
    .join(clientes.select('cliente_id', 'regiao', 'faixa_score'), 'cliente_id')
    .join(fretes.select('venda_id', 'valor_frete', 'prazo_dias', 'transportadora', 'status_entrega'), 'venda_id')
    .join(pagamentos.select('venda_id', 'metodo', 'parcelas'), 'venda_id')
    .withColumn(
        'valor_total',
        F.round(F.col('preco') * F.col('quantidade') * (1 - F.col('desconto_pct') / 100), 2)
    )
    .withColumn('ano_mes', F.date_format('data_venda', 'yyyy-MM'))
)
vendas.select('venda_id', 'nome', 'categoria', 'regiao', 'metodo',
             'valor_total', 'valor_frete', 'turno', 'ano_mes').show(10)
print('Vendas CONCLUIDAS:', vendas.count())

+--------+--------------+-----------+------------+--------------+-----------+-----------+-----+-------+
|venda_id|          nome|  categoria|      regiao|        metodo|valor_total|valor_frete|turno|ano_mes|
+--------+--------------+-----------+------------+--------------+-----------+-----------+-----+-------+
|       5|  Notebook 749|Eletrônicos|     Sudeste|           pix|    5509.76|       14.5|noite|2024-02|
|       6|Carregador 373|Eletrônicos|       Norte|        boleto|     3139.5|      24.53|tarde|2022-12|
|       8|    Câmera 309|Eletrônicos|     Sudeste|cartao_credito|    2538.19|      35.45|tarde|2022-09|
|       9|     Arroz 740|  Alimentos|Centro-Oeste|cartao_credito|     385.26|      41.14|manhã|2024-01|
|      10|     Calça 681|     Roupas|     Sudeste|           pix|     249.18|       44.8|tarde|2022-05|
|      12|  Notebook 591|Eletrônicos|Centro-Oeste|           pix|    11713.8|      45.62|noite|2024-04|
|      15|Smartphone 374|Eletrônicos|    Nordeste|cartao_credito

In [14]:
# Avaliações — tipa e enriquece com vendas e produtos
avaliacoes = (
    spark.read.format('delta').load(f'{BRONZE}/avaliacoes')
    .withColumn('data', F.col('data').cast(DateType()))
    .withColumn('nota', F.col('nota').cast('integer'))
    .filter(F.col('avaliacao_id').isNotNull())
    .join(
        vendas.select('venda_id', 'cliente_id', 'produto_id', 'valor_total', 'ano_mes'),
        'venda_id', 'left'
    )
    .join(
        produtos.select('produto_id', F.col('nome').alias('produto_nome'), 'categoria'),
        'produto_id', 'left'
    )
)
avaliacoes.select('avaliacao_id', 'produto_nome', 'categoria', 'nota', 'valor_total', 'data').show(10)
print('Avaliações:', avaliacoes.count())

+------------+------------------+----------+----+-----------+----------+
|avaliacao_id|      produto_nome| categoria|nota|valor_total|      data|
+------------+------------------+----------+----+-----------+----------+
|          10|          Pneu 794|Automotivo|   4|     1548.5|2022-03-10|
|       80282|      Película 108|Automotivo|   5|     749.58|2022-08-17|
|      158285|Porta-retratos 815|      Casa|   5|     232.02|2023-04-07|
|      235938|       Perfume 390|    Beleza|   3|     303.65|2024-06-19|
|      235941|       Arte de 870|    Livros|   4|     186.46|2023-07-10|
|      235946|        Spark: 954|    Livros|   4|     218.36|2023-10-13|
|      313634|           GPS 486|Automotivo|   5|     3107.7|2023-10-21|
|      313635|          Sofá 162|      Casa|   5|     235.88|2023-08-31|
|      313638| Princípios de 223|    Livros|   1|      70.42|2023-12-02|
|      391346|      Capacete 287|  Esportes|   4|    3697.25|2023-05-30|
+------------+------------------+----------+----+--

Avaliações: 420137


In [15]:
# Persiste na Silver
for name, df in [
    ('fornecedores', fornecedores), ('cupons', cupons),
    ('fretes', fretes), ('pagamentos', pagamentos),
    ('clientes', clientes), ('produtos', produtos),
    ('vendas', vendas), ('avaliacoes', avaliacoes),
]:
    df.write.format('delta').mode('overwrite').save(f'{SILVER}/{name}')
    print(f'[silver/{name}] salvo — {df.count()} linhas')

[silver/fornecedores] salvo — 50 linhas
[silver/cupons] salvo — 200 linhas


26/06/21 02:45:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/21 02:45:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/21 02:45:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/21 02:45:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/06/21 02:45:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/21 02:45:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/21 02:45:52 WARN MemoryManager: Total allocation exceeds 95.0

[silver/fretes] salvo — 799772 linhas


26/06/21 02:45:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/21 02:45:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/21 02:45:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/21 02:45:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/21 02:45:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


[silver/pagamentos] salvo — 600196 linhas


[silver/clientes] salvo — 100000 linhas
[silver/produtos] salvo — 1000 linhas


26/06/21 02:46:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/21 02:46:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/21 02:46:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/21 02:46:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/06/21 02:46:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/06/21 02:46:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/06/21 02:46:08 WARN MemoryManager: Total allocation exceeds 95.

[silver/vendas] salvo — 600196 linhas


26/06/21 02:46:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/21 02:46:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/21 02:46:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/21 02:46:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/06/21 02:46:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/06/21 02:46:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/06/21 02:46:20 WARN MemoryManager: Total allocation exceeds 95.

[silver/avaliacoes] salvo — 420137 linhas
